# 2강 실습: 전처리 — 흔들리지 않는 검사 만들기

**사용법**: 위에서부터 코드 칸을 클릭하고 `Shift + Enter`. 결과는 칸 아래에 나옵니다.

**오늘 할 것**
- 0부. 1강 문법 보충: `np.clip`, `plt.subplots`, `zip` 반복문
- 1부. 연습용 제품 사진 만들기 (금속 와셔/링 부품)
- 2부. 전처리 5가지: 크기 조절 → ROI 자르기 → 블러 → 이진화 → 밝기 정규화
- 3부. 🏭 미니 프로젝트: 위치·조명·노이즈가 바뀌어도 정확한 검사

> 코드 오른쪽의 `#` 뒤 글은 **설명(주석)** 입니다. 컴퓨터는 `#` 뒤를 무시하므로 마음껏 읽으세요.
> 그래프 제목은 영어로 씁니다. (Colab에 한글 글꼴이 없어 한글이 깨지기 때문)

## 준비: 라이브러리 불러오기

In [ ]:
import numpy as np                 # 숫자 표 계산 도구 (이름을 짧게 np라고 부름)
import cv2                         # OpenCV: 이미지 처리 도구
import matplotlib.pyplot as plt    # 그림 그리기 도구 (짧게 plt)

print("준비 완료!")

---
# 0부. 1강 문법 보충

1강 코드에서 자주 나온 3가지를 아주 작은 예제로 다시 봅니다.

## ① `np.clip(값, 최소, 최대)` — "범위 밖으로 못 나가게 자르기"

이미지 픽셀은 0~255만 가능합니다. 밝기를 +60 하면 250 → 310 처럼 255를 넘을 수 있죠.
`np.clip` 은 **최소보다 작으면 최소로, 최대보다 크면 최대로** 바꿉니다. 사이에 있는 값은 그대로 둡니다.

In [ ]:
numbers = np.array([-20, 0, 100, 250, 310])   # 숫자 5개짜리 배열(표) 만들기
clipped = np.clip(numbers, 0, 255)            # 0보다 작으면 0, 255보다 크면 255로

print("원래 값 :", numbers)    # [-20   0 100 250 310]
print("clip 후 :", clipped)    # [  0   0 100 250 255]  ← -20은 0, 310은 255가 됨

## ② `plt.subplots(행, 열)` — "그림판을 여러 칸으로 나누기"

- `fig` = 그림판 전체(종이 한 장)
- `axes` = 그 안의 칸들. 1행 3열이면 `axes[0]`, `axes[1]`, `axes[2]` 세 칸
- `figsize=(가로, 세로)` = 종이 크기(인치)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(9, 3))   # 1줄에 3칸짜리 그림판 만들기

axes[0].plot([1, 2, 3], [1, 4, 9])     # 첫째 칸(0번)에 선 그래프
axes[0].set_title("Box 0")              # 첫째 칸 제목

axes[1].bar(["A", "B"], [3, 5])         # 둘째 칸(1번)에 막대 그래프
axes[1].set_title("Box 1")

axes[2].imshow(np.eye(5), cmap="gray")  # 셋째 칸(2번)에 5x5 이미지 (대각선만 흰색)
axes[2].set_title("Box 2")

plt.show()                              # 화면에 보여주기

## ③ `for a, b in zip(목록1, 목록2)` — "두 목록을 짝지어 하나씩 꺼내기"

`zip` 은 지퍼처럼 두 목록을 **같은 순서끼리 짝**지어 줍니다.
1강에서 `for a, (name, im) in zip(axes, versions.items())` 는 "**그림 칸 하나 + (이름, 사진) 하나**"를 짝지어 꺼낸 것입니다.

In [ ]:
boxes  = ["칸0", "칸1", "칸2"]
photos = ["원본", "밝게", "대비"]

for box, photo in zip(boxes, photos):     # (칸0, 원본) → (칸1, 밝게) → (칸2, 대비) 순서로 꺼냄
    print(box, "에", photo, "사진을 그린다")

1강 코드도 똑같은 구조입니다. 그림 칸 3개와 사진 3장을 짝지어서 칸마다 사진 하나를 그립니다.

`ravel()` 은 2차원 표를 **한 줄로 쭉 펴는** 것, `hist` 는 **값별 개수를 막대그래프**로 그리는 것입니다.

In [ ]:
tiny = np.array([[0, 0, 255],
                 [0, 128, 255]])          # 2줄 x 3칸짜리 작은 '이미지'
print("펴기 전 모양:", tiny.shape)          # (2, 3)
print("ravel() 후 :", tiny.ravel())        # [  0   0 255   0 128 255]  ← 한 줄로 펴짐

plt.hist(tiny.ravel(), bins=256, range=(0, 256))   # 0이 3개, 128이 1개, 255가 2개 → 막대 3개
plt.title("Histogram of tiny image")
plt.show()

---
# 1부. 연습용 제품 사진 만들기

회사 사진은 쓸 수 없으니, **금속 링(와셔) 부품**을 컨베이어 위에서 찍은 것 같은 가짜 사진을 만듭니다.
이 함수 하나로 위치·조명·노이즈·불량을 마음대로 바꿀 수 있습니다.

| 설정 | 뜻 | 예 |
|---|---|---|
| `dx`, `dy` | 제품이 가운데에서 가로/세로로 몇 픽셀 벗어났나 | `dx=25` → 오른쪽으로 25 |
| `light` | 조명 밝기 변화 | `light=40` → 전체 +40 밝게 |
| `noise` | 센서 잡음 세기 | 클수록 자글자글 |
| `dust` | 먼지 점(흰·검은 점) 비율 | `0.02` → 2% 픽셀 |
| `scratch`, `dent` | 긁힘 / 찍힘 불량 넣기 | `True` 면 넣음 |

> 이 칸은 "사진 만드는 공장"이라 내용을 다 이해하지 않아도 됩니다. 실행만 하세요.

In [ ]:
rng = np.random.default_rng(seed=1)   # 난수 생성기 (seed 고정 → 매번 같은 결과)
H, W = 360, 480                       # 사진 크기: 세로 360, 가로 480 픽셀

def make_part_photo(dx=0, dy=0, light=0, noise=4, dust=0.0, scratch=False, dent=False):
    img = np.full((H, W), 60, dtype=np.float32)          # 어두운 컨베이어 배경(밝기 60)
    img += rng.normal(0, 4, (H, 1)) * np.ones((1, W))    # 컨베이어 벨트 가로 줄무늬
    c = (W // 2 + dx, H // 2 + dy)                       # 제품 중심 위치 (가운데 + 이동량)
    cv2.circle(img, c, 90, 185, -1)                      # 금속 링 바깥 원 (반지름 90, 밝기 185)
    cv2.circle(img, c, 30, 60, -1)                       # 가운데 구멍 (반지름 30, 배경색)
    if scratch:                                          # 긁힘 불량: 어두운 선
        cv2.line(img, (c[0] + 40, c[1] - 50), (c[0] + 70, c[1] - 10), 120, 2)
    if dent:                                             # 찍힘 불량: 작은 어두운 점
        cv2.circle(img, (c[0] - 55, c[1] + 30), 5, 110, -1)
    img += light                                         # 조명 변화
    img += rng.normal(0, noise, img.shape)               # 센서 잡음
    img = np.clip(img, 0, 255).astype(np.uint8)          # 0~255로 자르고 이미지 형식으로
    if dust > 0:                                         # 먼지: 흰 점·검은 점 뿌리기
        r = rng.random(img.shape)                        # 픽셀마다 0~1 사이 난수
        img[r < dust / 2] = 0                            # 일부는 검은 점
        img[r > 1 - dust / 2] = 255                      # 일부는 흰 점
    return img

reference = make_part_photo()                            # 기준이 될 양품 사진
print("사진 크기:", reference.shape)

plt.figure(figsize=(6, 4.5))
plt.imshow(reference, cmap="gray", vmin=0, vmax=255)     # vmin/vmax: 0=검정, 255=흰색으로 고정
plt.title("Reference: good metal ring")
plt.axis("off")                                          # 눈금 숨기기
plt.show()

### 현장에서 생기는 다양한 사진 미리 보기

In [ ]:
samples = {
    "Good (normal)":           make_part_photo(),
    "Good, shifted":           make_part_photo(dx=30, dy=-20),
    "Good, brighter + dust":   make_part_photo(light=40, dust=0.02),
    "BAD: scratch":            make_part_photo(scratch=True),
    "BAD: dent, darker":       make_part_photo(light=-30, dent=True),
    "BAD: scratch, shifted":   make_part_photo(dx=-25, dy=15, scratch=True),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))          # 2줄 x 3칸 그림판
for ax, (name, im) in zip(axes.ravel(), samples.items()): # axes.ravel(): 2x3 칸을 한 줄(6칸)로 펴기
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.show()

---
# 2부. 전처리 5가지

## ① 크기 조절 (Resize)

`cv2.resize(사진, (새 가로, 새 세로))` — 순서가 **(가로, 세로)** 인 것에 주의하세요. (`shape` 는 (세로, 가로))

너무 작게 줄이면 긁힘이 사라지는 것을 확인해 보세요.

In [ ]:
bad = make_part_photo(scratch=True)                          # 긁힘 있는 사진

half  = cv2.resize(bad, (W // 2, H // 2))                    # 1/2 크기 (240 x 180)
tiny  = cv2.resize(bad, (W // 10, H // 10))                  # 1/10 크기 (48 x 36)

print("원본 :", bad.shape, "/ 1/2 :", half.shape, "/ 1/10 :", tiny.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, im) in zip(axes, [("Original 480x360", bad), ("1/2: 240x180", half), ("1/10: 48x36", tiny)]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis("off")
plt.show()
print("→ 1/10 크기에서는 긁힘이 거의 안 보입니다. 가장 작은 불량이 3~5픽셀 이상 남도록 줄여야 합니다.")

## ② ROI 자르기 — 검사할 부분만

`사진[위:아래, 왼쪽:오른쪽]` 으로 자릅니다. 제품 중심이 (240, 180) 이고 반지름이 90이니, 중심에서 ±100 픽셀을 자르면 제품이 딱 들어옵니다.

In [ ]:
cx, cy = W // 2, H // 2                     # 제품 중심 (가로 240, 세로 180)
R = 100                                     # 중심에서 위아래·좌우로 100픽셀씩

roi = reference[cy - R : cy + R,            # 세로: 80 ~ 280
                cx - R : cx + R]            # 가로: 140 ~ 340
print("ROI 크기:", roi.shape)                # (200, 200)

plt.figure(figsize=(4, 4))
plt.imshow(roi, cmap="gray", vmin=0, vmax=255)
plt.title("ROI 200x200")
plt.axis("off")
plt.show()

**문제**: 제품이 옆으로 30픽셀 밀려 있으면? 같은 위치를 자르면 제품이 잘립니다. ↓

In [ ]:
shifted = make_part_photo(dx=30, dy=-20)                    # 오른쪽 30, 위로 20 밀린 제품
roi_wrong = shifted[cy - R : cy + R, cx - R : cx + R]       # 고정 위치로 자르기

plt.figure(figsize=(4, 4))
plt.imshow(roi_wrong, cmap="gray", vmin=0, vmax=255)
plt.title("Fixed ROI on shifted part: cut off!")
plt.axis("off")
plt.show()
print("→ 제품 위치를 먼저 찾아야 합니다. ④ 이진화에서 해결합니다.")

## ③ 블러 — 노이즈 지우기

먼지(흰·검은 점)가 많은 사진에 두 가지 블러를 적용해 비교합니다.
- `cv2.GaussianBlur(사진, (5, 5), 0)` : 5×5 주변을 가중 평균
- `cv2.medianBlur(사진, 5)` : 5×5 주변의 **중간값** → 튀는 점에 특히 강함

✏️ 숫자 5를 3, 9, 15로 바꿔 보세요. 숫자가 클수록 더 흐려지고, 긁힘도 같이 흐려집니다. (홀수만 가능)

In [ ]:
dusty = make_part_photo(dust=0.03, scratch=True)      # 먼지 3% + 긁힘

KSIZE = 5                                             # ✏️ 블러 크기 (홀수: 3, 5, 7, 9 ...)
gauss  = cv2.GaussianBlur(dusty, (KSIZE, KSIZE), 0)   # 가우시안 블러
median = cv2.medianBlur(dusty, KSIZE)                 # 미디언 블러

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, im) in zip(axes, [("Dusty photo", dusty), (f"Gaussian {KSIZE}x{KSIZE}", gauss), (f"Median {KSIZE}x{KSIZE}", median)]):
    ax.imshow(im[60:300, 120:360], cmap="gray", vmin=0, vmax=255)   # 제품 부분만 확대해서 보기
    ax.set_title(name)
    ax.axis("off")
plt.show()
print("→ 미디언 블러가 먼지 점을 훨씬 깨끗하게 지우고, 긁힘은 남깁니다.")

## ④ 이진화 — 제품과 배경 나누기 + 제품 위치 찾기

`cv2.threshold(사진, 기준값, 255, 방식)` → 기준보다 밝으면 255(흰색), 아니면 0(검정)

같은 제품을 **보통 / 어두운 조명 / 밝은 조명**으로 찍고 두 방법을 비교합니다.
- 고정 임계값 120 (사람이 정함)
- Otsu 자동 임계값 (`cv2.THRESH_OTSU`, 기준값 자리에 0을 넣으면 자동 계산)

In [ ]:
lights = {"Normal": 0, "Dark (-70)": -70, "Bright (+70)": 70}

fig, axes = plt.subplots(3, 3, figsize=(13, 11))
for row, (name, lv) in enumerate(lights.items()):      # enumerate: 순번(0,1,2)과 값을 같이 꺼냄
    photo = make_part_photo(light=lv)
    _, fixed = cv2.threshold(photo, 120, 255, cv2.THRESH_BINARY)                     # 고정 120
    otsu_value, otsu = cv2.threshold(photo, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # 자동

    axes[row, 0].imshow(photo, cmap="gray", vmin=0, vmax=255); axes[row, 0].set_title(f"{name} photo")
    axes[row, 1].imshow(fixed, cmap="gray");  axes[row, 1].set_title("Fixed threshold 120")
    axes[row, 2].imshow(otsu, cmap="gray");   axes[row, 2].set_title(f"Otsu (auto = {otsu_value:.0f})")
    print(f"{name:13s}: Otsu가 자동으로 고른 기준값 = {otsu_value:.0f}")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

→ 고정 120은 어두우면 제품이 사라지고, 밝으면 배경까지 흰색이 됩니다. **Otsu는 조명에 맞춰 기준값을 스스로 바꿉니다.**

### 이진화 결과로 제품 중심 찾기

흰 덩어리(제품)의 **무게중심**을 계산하면 제품 위치를 알 수 있습니다.
- `cv2.findContours` : 흰 덩어리들의 외곽선 찾기
- `max(..., key=cv2.contourArea)` : 그중 가장 큰 덩어리 = 제품
- `cv2.moments` : 덩어리의 무게중심 계산 (m10/m00 = 가로 중심, m01/m00 = 세로 중심)

In [ ]:
def find_center(gray):
    _, mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)  # Otsu 이진화
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # 바깥 외곽선들
    biggest = max(contours, key=cv2.contourArea)       # 면적이 가장 큰 것 = 제품
    M = cv2.moments(biggest)                           # 무게중심 계산용 값들
    cx = int(round(M["m10"] / M["m00"]))               # 가로 중심
    cy = int(round(M["m01"] / M["m00"]))               # 세로 중심
    return cx, cy

found_x, found_y = find_center(shifted)                # 아까 밀려 있던 사진
print("찾은 중심:", (found_x, found_y), "/ 실제 중심:", (W // 2 + 30, H // 2 - 20))

roi_right = shifted[found_y - R : found_y + R, found_x - R : found_x + R]   # 찾은 중심으로 자르기
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(roi_wrong, cmap="gray", vmin=0, vmax=255); axes[0].set_title("Fixed ROI (wrong)")
axes[1].imshow(roi_right, cmap="gray", vmin=0, vmax=255); axes[1].set_title("ROI at found center (right)")
for ax in axes:
    ax.axis("off")
plt.show()

## ⑤ 밝기 정규화 — 조명 차이 없애기

공식: `새 값 = 원래 값 − 평균 밝기 + 128` → 어떤 조명이든 평균이 128로 맞춰집니다.

In [ ]:
def normalize(gray):
    g = gray.astype(np.float32)               # 빼기 계산 중 음수가 나올 수 있어 실수형으로
    g = g - g.mean() + 128                    # 평균을 빼고 128을 더함 → 평균이 128이 됨
    return np.clip(g, 0, 255).astype(np.uint8)  # 0~255로 자르고 이미지 형식으로

dark   = make_part_photo(light=-40)
bright = make_part_photo(light=40)
print("정규화 전 평균 밝기:  어두운 사진", round(dark.mean()), "/ 밝은 사진", round(bright.mean()))
print("정규화 후 평균 밝기:  어두운 사진", round(normalize(dark).mean()), "/ 밝은 사진", round(normalize(bright).mean()))

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes[0, 0].imshow(dark,   cmap="gray", vmin=0, vmax=255); axes[0, 0].set_title("Dark: before")
axes[0, 1].imshow(bright, cmap="gray", vmin=0, vmax=255); axes[0, 1].set_title("Bright: before")
axes[1, 0].imshow(normalize(dark),   cmap="gray", vmin=0, vmax=255); axes[1, 0].set_title("Dark: after")
axes[1, 1].imshow(normalize(bright), cmap="gray", vmin=0, vmax=255); axes[1, 1].set_title("Bright: after")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

---
# 3부. 🏭 미니 프로젝트: 흔들리지 않는 검사

1강의 단순 검사 vs 2강의 전처리 검사를 **같은 시험 사진 6장**으로 비교합니다.

```
사진 → ① 미디언+가우시안 블러 → ② Otsu로 제품 중심 찾기 → ③ 중심 기준 ROI 자르기
     → ④ 밝기 정규화(링 표면 평균 기준) → ⑤ 양품 사진과 빼기 → ⑥ 검사 마스크 안에서만 판정
```

### 검사 마스크

링의 **테두리**는 위치가 1픽셀만 어긋나도 큰 차이가 납니다(밝은 링 ↔ 어두운 배경).
그래서 테두리를 뺀 **링 표면 안쪽만** 흰색인 마스크를 만들어 거기만 검사합니다.

In [ ]:
def make_inspection_mask():
    mask = np.zeros((2 * R, 2 * R), dtype=np.uint8)    # 200x200 검정 판
    cv2.circle(mask, (R, R), 82, 255, -1)              # 바깥 원(반지름 82)을 흰색으로 칠하기 (링 바깥 90보다 8 작게)
    cv2.circle(mask, (R, R), 38, 0, -1)                # 안쪽 원(반지름 38)을 다시 검정으로 (구멍 30보다 8 크게)
    return mask                                        # → 테두리 8픽셀씩 뺀 '도넛' 모양

INSPECT_MASK = make_inspection_mask()
plt.figure(figsize=(4, 4))
plt.imshow(INSPECT_MASK, cmap="gray")
plt.title("Inspection mask (white = inspect)")
plt.axis("off")
plt.show()

### 전처리 함수와 검사 함수

In [ ]:
def preprocess(photo):
    g = cv2.medianBlur(photo, 5)                       # ① 먼지 점 제거
    g = cv2.GaussianBlur(g, (5, 5), 0)                 # ① 잔잔한 잡음 제거
    cx, cy = find_center(g)                            # ② 제품 중심 찾기
    roi = g[cy - R : cy + R, cx - R : cx + R]          # ③ 중심 기준 200x200 자르기
    roi = roi.astype(np.float32)
    surface_mean = roi[INSPECT_MASK > 0].mean()        # ④ 링 표면(마스크 흰 부분)의 평균 밝기
    roi = roi - surface_mean + 128                     # ④ 표면 평균을 128로 맞춤
    return np.clip(roi, 0, 255).astype(np.uint8)

THRESHOLD = 25    # ✏️ 차이가 이보다 크면 불량 후보
MIN_AREA  = 10    # ✏️ 이보다 작은 덩어리는 무시 (픽셀 수)

def inspect_v2(photo, reference):
    ref_roi  = preprocess(reference)                   # 기준 사진 전처리
    test_roi = preprocess(photo)                       # 검사 사진 전처리
    diff = cv2.absdiff(ref_roi, test_roi)              # ⑤ 두 사진 차이
    diff = cv2.bitwise_and(diff, diff, mask=INSPECT_MASK)   # ⑥ 마스크 밖(테두리·배경)은 0으로 지움
    _, bad_pixels = cv2.threshold(diff, THRESHOLD, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(bad_pixels, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    defects = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]   # 작은 점은 버리기

    result = cv2.cvtColor(test_roi, cv2.COLOR_GRAY2RGB)                 # 빨간 박스용 컬러 변환
    for c in defects:
        x, y, w, h = cv2.boundingRect(c)                                # 덩어리를 감싸는 네모
        cv2.rectangle(result, (x - 3, y - 3), (x + w + 3, y + h + 3), (255, 0, 0), 2)
    return ("NG" if defects else "OK"), result

def inspect_v1(photo, reference):
    """1강 방식: 전처리 없이 그대로 빼기"""
    diff = cv2.absdiff(reference, photo)
    _, bad_pixels = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(bad_pixels, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    defects = [c for c in contours if cv2.contourArea(c) >= MIN_AREA]
    return "NG" if defects else "OK"

print("함수 준비 완료")

### 시험: 정답을 알고 있는 사진 6장

| 사진 | 정답 |
|---|---|
| 양품 / 양품+이동+밝음+먼지 / 양품+이동+어두움 | **OK** |
| 긁힘 / 긁힘+이동+밝음+먼지 / 찍힘+이동+어두움 | **NG** |

In [ ]:
tests = [
    ("Good",                       "OK", make_part_photo()),
    ("Good+shift+bright+dust",     "OK", make_part_photo(dx=25, dy=-15, light=40, dust=0.02)),
    ("Good+shift+dark",            "OK", make_part_photo(dx=-30, dy=20, light=-35)),
    ("Scratch",                    "NG", make_part_photo(scratch=True)),
    ("Scratch+shift+bright+dust",  "NG", make_part_photo(dx=25, dy=-15, light=40, dust=0.02, scratch=True)),
    ("Dent+shift+dark",            "NG", make_part_photo(dx=-20, light=-30, dent=True)),
]

print(f"{'사진':28s} {'정답':4s} {'1강 방식':8s} {'2강 방식':8s}")
score_v1 = score_v2 = 0                                        # 맞힌 개수 세기
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, (name, answer, photo) in zip(axes.ravel(), tests):
    v1 = inspect_v1(photo, reference)                          # 1강 방식 판정
    v2, result = inspect_v2(photo, reference)                  # 2강 방식 판정 + 결과 그림
    score_v1 += (v1 == answer)                                 # 맞으면 1 더하기 (True=1, False=0)
    score_v2 += (v2 == answer)
    mark1 = "O" if v1 == answer else "X"
    mark2 = "O" if v2 == answer else "X"
    print(f"{name:28s} {answer:4s} {v1} ({mark1})   {v2} ({mark2})")
    ax.imshow(result)
    ax.set_title(f"{name}\nanswer {answer} / v2 says {v2}")
    ax.axis("off")
plt.show()

print(f"\n정확도  1강 방식: {score_v1}/6   2강 방식: {score_v2}/6")

---
## ✅ 2강 정리

- **Resize**: 빠르게, 하지만 가장 작은 불량이 3~5픽셀은 남게.
- **ROI**: 제품 중심을 찾아서 그 주변만 자르면 위치가 달라도 OK.
- **Blur**: 먼지 점엔 미디언, 일반 잡음엔 가우시안. 너무 세면 불량도 지워짐.
- **Threshold**: 조명이 바뀌면 고정값은 실패 → **Otsu 자동 임계값**.
- **정규화**: 평균 밝기를 128로 맞추면 조명 차이가 사라짐.
- **검사 마스크**: 테두리처럼 원래 차이가 큰 곳은 빼고 검사.

> 🏭 회사 적용 포인트: 실제 검사기 설정 화면의 "검사 영역", "위치 보정", "감도(임계값)", "최소 불량 크기"가 바로 오늘 배운 것들입니다.

## 📝 연습문제

1. `THRESHOLD` 를 10, 50, 80으로 바꿔 보세요. 어느 값부터 양품이 NG(과검)가 되고, 어느 값부터 불량이 OK(미검)가 되나요?
2. 시험 사진에 `make_part_photo(dust=0.08, scratch=True)` (먼지 8%)를 추가해 보세요. 여전히 맞히나요? 틀린다면 `preprocess` 의 `medianBlur` 크기를 7로 바꿔 보세요.
3. `make_part_photo(dx=150)` 처럼 제품이 사진 밖으로 반쯤 나가면 어떤 오류가 나나요? 현장에서는 이런 경우 어떻게 처리하면 좋을까요?
4. 생각해 보기: 우리 회사 검사 사진에서 **ROI(검사 영역)** 를 잡는다면 어디일까요? 테두리처럼 "원래 차이가 큰 곳"은 어디일까요?

모르는 부분은 칸 번호와 함께 Claude에게 물어보세요. 예: "2강 3부 preprocess 함수 다시 설명해줘"